In [1]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import requests

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

Log loaded. Rows: 8
PROJECT_ROOT: /Users/boulanger/Documents/governance-framework


## UCDP Pipeline

**Source:** Uppsala Conflict Data Program — Georeferenced Event Dataset (GED)
**Access:** Automated via UCDP REST API — no registration required
**Download instructions:** See `docs/instructions_data_maintenance.md` — UCDP section

### Framework usage
| Indicator | Concept | Role |
|-----------|---------|------|
| Battle-related deaths | Political stability | Primary tier 1 |
| One-sided violence fatalities | Political stability | Primary tier 1 |
| Non-state conflict fatalities | Political stability | Primary tier 1 |

In [13]:
import requests
import zipfile
import io
import pandas as pd
from datetime import datetime

UCDP_BASE = "https://ucdp.uu.se/downloads"

# Updated URL patterns — all now ZIP files
UCDP_DATASETS = {
    'orgviolence_cy': ('organizedviolencecy', 'organizedviolencecy', '-csv.zip'),  # Country-year aggregate
    'armed_conflict':  ('ucdpprio', 'ucdp-prio-acd', '-csv.zip'),                 # Armed conflict
    'onesided':        ('nsos', 'ucdp-onesided', '-csv.zip'),                      # One-sided violence
}

def get_latest_ucdp_version():
    """Try recent versions to find latest available UCDP datasets."""
    current_year = datetime.today().year
    for year in range(current_year - 2000, (current_year - 2000) - 3, -1):
        version_str = f"{year}1"
        folder, prefix, suffix = UCDP_DATASETS['armed_conflict']
        url = f"{UCDP_BASE}/{folder}/{prefix}-{version_str}{suffix}"
        response = requests.head(url, timeout=10, allow_redirects=True)
        if response.status_code == 200 and 'text/html' not in response.headers.get('Content-Type', ''):
            print(f"Latest UCDP version: {year}.1 (code: {version_str})")
            return version_str
    return None

def download_ucdp_zip(folder, prefix, version, suffix):
    """Download a UCDP ZIP dataset and return as DataFrame."""
    url = f"{UCDP_BASE}/{folder}/{prefix}-{version}{suffix}"
    print(f"Downloading: {url}")
    response = requests.get(url, timeout=60)
    if response.status_code == 200 and 'text/html' not in response.headers.get('Content-Type', ''):
        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            csv_files = [f for f in z.namelist() if f.endswith('.csv')]
            print(f"  CSV files in ZIP: {csv_files}")
            df = pd.read_csv(z.open(csv_files[0]))
            print(f"  OK: {df.shape}")
            return df
    else:
        print(f"  Failed: {response.status_code}")
        return None

UCDP_VERSION = get_latest_ucdp_version()
print(f"\nUsing version: {UCDP_VERSION}")

Latest UCDP version: 25.1 (code: 251)

Using version: 251


In [14]:
dfs = {}
for name, (folder, prefix, suffix) in UCDP_DATASETS.items():
    df = download_ucdp_zip(folder, prefix, UCDP_VERSION, suffix)
    if df is not None:
        dfs[name] = df

print(f"\nDownloaded {len(dfs)} datasets")
for name, df in dfs.items():
    print(f"  {name}: {df.shape}")
    print(f"    Columns: {list(df.columns[:8])}")

Downloading: https://ucdp.uu.se/downloads/organizedviolencecy/organizedviolencecy-251-csv.zip
  CSV files in ZIP: ['organizedviolencecy_v25_1.csv']
  OK: (6936, 74)
Downloading: https://ucdp.uu.se/downloads/ucdpprio/ucdp-prio-acd-251-csv.zip
  CSV files in ZIP: ['UcdpPrioConflict_v25_1.csv']
  OK: (2752, 28)
Downloading: https://ucdp.uu.se/downloads/nsos/ucdp-onesided-251-csv.zip
  CSV files in ZIP: ['OneSided_v25_1.csv']
  OK: (1330, 17)

Downloaded 3 datasets
  orgviolence_cy: (6936, 74)
    Columns: ['country_id_cy', 'country_cy', 'year_cy', 'region_cy', 'main_govt_name_cy', 'sb_exist_cy', 'sb_dyad_count_cy', 'sb_dyad_ids_cy']
  armed_conflict: (2752, 28)
    Columns: ['conflict_id', 'location', 'side_a', 'side_a_id', 'side_a_2nd', 'side_b', 'side_b_id', 'side_b_2nd']
  onesided: (1330, 17)
    Columns: ['conflict_id', 'dyad_id', 'actor_id', 'coalition_components', 'actor_name', 'actor_name_fulltext', 'actor_name_mothertongue', 'year']


In [15]:
# Inspect country-year dataset columns
cy = dfs['orgviolence_cy']
print(f"Shape: {cy.shape}")
print(f"Years: {cy['year_cy'].min()} — {cy['year_cy'].max()}")
print(f"Countries: {cy['country_cy'].nunique()}")
print("\nAll columns:")
for col in cy.columns:
    print(f"  {col}")

Shape: (6936, 74)
Years: 1989 — 2024
Countries: 199

All columns:
  country_id_cy
  country_cy
  year_cy
  region_cy
  main_govt_name_cy
  sb_exist_cy
  sb_dyad_count_cy
  sb_dyad_ids_cy
  sb_dyad_names_cy
  sb_deaths_parties_cy
  sb_deaths_civilians_cy
  sb_deaths_unknown_cy
  sb_total_deaths_best_cy
  sb_total_deaths_high_cy
  sb_total_deaths_low_cy
  sb_intrastate_exist_cy
  sb_intrastate_dyad_count_cy
  sb_intrastate_dyad_ids_cy
  sb_intrastate_dyad_names_cy
  sb_intrastate_main_govt_inv_incomp_cy
  sb_intrastate_deaths_parties_cy
  sb_intrastate_deaths_civilians_cy
  sb_intrastate_deaths_unknown_cy
  sb_intrastate_deaths_best_cy
  sb_intrastate_deaths_high_cy
  sb_intrastate_deaths_low_cy
  sb_interstate_exist_cy
  sb_interstate_dyad_count_cy
  sb_interstate_dyad_ids_cy
  sb_interstate_dyad_names_cy
  sb_interstate_main_govt_inv_incomp_cy
  sb_interstate_deaths_parties_cy
  sb_interstate_deaths_civilians_cy
  sb_interstate_deaths_unknown_cy
  sb_interstate_deaths_best_cy
  sb_inte

In [16]:
# Select framework-relevant columns
KEEP_COLS = [
    'country_id_cy', 'country_cy', 'year_cy',
    # State-based conflict
    'sb_exist_cy', 'sb_dyad_count_cy', 'sb_total_deaths_best_cy',
    'sb_intrastate_exist_cy', 'sb_intrastate_deaths_best_cy',
    'sb_interstate_exist_cy', 'sb_interstate_deaths_best_cy',
    # Non-state conflict
    'ns_exist_cy', 'ns_dyad_count_cy', 'ns_total_deaths_best_cy',
    # One-sided violence
    'os_exist_cy', 'os_total_deaths_best_cy', 'os_any_govt_killings_best_cy',
    # Total
    'cumulative_total_deaths_in_orgvio_best_cy',
]

ucdp = cy[KEEP_COLS].copy()

# Rename columns
ucdp = ucdp.rename(columns={
    'country_id_cy':                        'country_id',
    'country_cy':                           'country_name',
    'year_cy':                              'year',
    'sb_exist_cy':                          'ucdp_sb_conflict_exists',
    'sb_dyad_count_cy':                     'ucdp_sb_conflict_count',
    'sb_total_deaths_best_cy':              'ucdp_sb_deaths_best',
    'sb_intrastate_exist_cy':               'ucdp_sb_intrastate_exists',
    'sb_intrastate_deaths_best_cy':         'ucdp_sb_intrastate_deaths_best',
    'sb_interstate_exist_cy':               'ucdp_sb_interstate_exists',
    'sb_interstate_deaths_best_cy':         'ucdp_sb_interstate_deaths_best',
    'ns_exist_cy':                          'ucdp_ns_conflict_exists',
    'ns_dyad_count_cy':                     'ucdp_ns_conflict_count',
    'ns_total_deaths_best_cy':              'ucdp_ns_deaths_best',
    'os_exist_cy':                          'ucdp_os_violence_exists',
    'os_total_deaths_best_cy':              'ucdp_os_deaths_best',
    'os_any_govt_killings_best_cy':         'ucdp_os_govt_killings_best',
    'cumulative_total_deaths_in_orgvio_best_cy': 'ucdp_total_orgvio_deaths_best',
})

# Filter to framework start year
ucdp = ucdp[ucdp['year'] >= FRAMEWORK_START_YEAR].copy()
ucdp = ucdp.sort_values(['country_name', 'year']).reset_index(drop=True)

print(f"Shape: {ucdp.shape}")
print(f"Years: {ucdp['year'].min()} — {ucdp['year'].max()}")
print(f"Countries: {ucdp['country_name'].nunique()}")
print(f"\nMissing values (%):")
missing_pct = (ucdp.isnull().sum() / len(ucdp) * 100).round(1)
print(missing_pct[missing_pct > 0].sort_values(ascending=False))
print(ucdp.head())

Shape: (6764, 17)
Years: 1990 — 2024
Countries: 199

Missing values (%):
Series([], dtype: float64)
   country_id country_name  year  ucdp_sb_conflict_exists  \
0         700  Afghanistan  1990                        1   
1         700  Afghanistan  1991                        1   
2         700  Afghanistan  1992                        1   
3         700  Afghanistan  1993                        1   
4         700  Afghanistan  1994                        1   

   ucdp_sb_conflict_count  ucdp_sb_deaths_best  ucdp_sb_intrastate_exists  \
0                       5                 1478                          1   
1                       4                 3302                          1   
2                       4                 4287                          1   
3                       4                 4071                          1   
4                       3                 8937                          1   

   ucdp_sb_intrastate_deaths_best  ucdp_sb_interstate_exists  \
0     

In [17]:
# Save to processed
output_path = os.path.join(PROCESSED_DIR, "ucdp_clean.csv")
ucdp.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {ucdp.shape}")

# Update download log
update_entry(
    "UCDP",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=str(int(ucdp['year'].max())),
    local_filename="ucdp_clean.csv",
    latest_available_version=f"25.1",
    notes=f"Country-year organized violence dataset. 16 indicators across state-based, non-state, and one-sided violence. Auto-detects latest version. Coverage: 1989-2024, 199 countries."
)

print_entry("UCDP")

Written: /Users/boulanger/Documents/governance-framework/data/processed/ucdp_clean.csv
Shape: (6764, 17)
[download_log] Updated entry for UCDP
  source_id: UCDP
  last_attempted_date: 2026-06-01
  last_successful_download_date: 2026-06-01
  data_as_of_date: 2024
  local_filename: ucdp_clean.csv
  latest_available_version: 25.1
  no_update_reason: nan
  notes: Country-year organized violence dataset. 16 indicators across state-based, non-state, and one-sided violence. Auto-detects latest version. Coverage: 1989-2024, 199 countries.
